## 1. Deploying Machine Learning Models - Intro

In this chapter we will take the churn prediction model we trained earlier and turn it into a service that other systems can call over the network.

### The problem: the model lives in a notebook

In chapter 3 we trained a model that predicts whether a customer of a telecom company will churn (stop using the service).  
The model is useful only if people can actually use it.  
Right now it lives inside a notebook: to get a prediction, we open the notebook, run all the cells, and look at the output. That is fine for us, but the rest of the company cannot work this way.

Imagine the marketing team wants to send a promo email to customers who are likely to leave.  
Their system needs to ask the model "what is the churn probability of this customer?", automatically, for thousands of customers, without anyone opening a notebook.

The solution is to deploy the model: run it on a server and expose an API endpoint.  
Other services (like the marketing service) send a request with the customer information to this endpoint, and get the prediction back in the response. Based on that response, the marketing service can decide to send a promo email or do nothing.

<img src="https://github.com/k-candidate/machine-learning-zoomcamp/raw/main/05-deployment/images/01-intro-02-model-deployment-diagram-imagegen.jpg" alt="ML Model Deployment Diagram" style="max-width: 100%; height: auto; display: block; margin: 0 auto;">

### The plan for this chapter

We will take the churn model from the notebook to a working web service step by step:
- Saving and loading the model with pickle, and turning the notebook into a Python script, so the model can be used without re-training it every time.
- Creating a web service with FastAPI: first a small ping/pong service, then the actual churn service that answers prediction requests.
- Preparing the service for production with uvicorn.
- Managing the project dependencies with uv, so the service runs with the library versions we tested it with.
- Packaging everything into a Docker container, so it can run the same way on any machine.

Each step removes one thing that ties the model to our laptop: the notebook, the development server, the system-wide Python packages, the operating system and finally the local machine itself.

## 2. Saving and loading the model

In this section we save the trained churn model to a file with pickle, load it back, and turn the notebook into a Python script, so the model can be used later without training it again.

### Why we need to save the model

**Training gives us two objects:**
- **the `DictVectorizer` that turns customer records into a feature matrix**
- **the logistic regression model itself.**

**Both live in the memory of the notebook process. Close (or restart) the notebook and they are gone.**

Re-training the model every time we want a prediction is wasteful: training takes time, and we would get a slightly different model depending on the data split and random state. What we want instead is to train once, save the result to a file, and later simply load that file in the web service.

In [21]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold

from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

from sklearn.pipeline import make_pipeline

import pickle

In [22]:
data_url = 'https://raw.githubusercontent.com/k-candidate/mlbookcamp-code/master/chapter-03-churn-prediction/WA_Fn-UseC_-Telco-Customer-Churn.csv'

df = pd.read_csv(data_url)

df.columns = df.columns.str.lower().str.replace(' ', '_')

categorical_columns = list(df.dtypes[df.dtypes == 'str'].index)

for c in categorical_columns:
    df[c] = df[c].str.lower().str.replace(' ', '_')

df.totalcharges = pd.to_numeric(df.totalcharges, errors='coerce')
df.totalcharges = df.totalcharges.fillna(0)

df.churn = (df.churn == 'yes').astype(int)

In [23]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)

In [24]:
numerical = ['tenure', 'monthlycharges', 'totalcharges']

categorical = [
    'gender',
    'seniorcitizen',
    'partner',
    'dependents',
    'phoneservice',
    'multiplelines',
    'internetservice',
    'onlinesecurity',
    'onlinebackup',
    'deviceprotection',
    'techsupport',
    'streamingtv',
    'streamingmovies',
    'contract',
    'paperlessbilling',
    'paymentmethod',
]

In [25]:
def train(df_train, y_train, C=1.0):
    dicts = df_train[categorical + numerical].to_dict(orient='records')

    dv = DictVectorizer(sparse=False)
    X_train = dv.fit_transform(dicts)

    model = LogisticRegression(C=C, max_iter=3000)
    model.fit(X_train, y_train)
    
    return dv, model

In [26]:
def predict(df, dv, model):
    dicts = df[categorical + numerical].to_dict(orient='records')

    X = dv.transform(dicts)
    y_pred = model.predict_proba(X)[:, 1]

    return y_pred

In [27]:
C = 1.0
n_splits = 5

In [28]:
kfold = KFold(n_splits=n_splits, shuffle=True, random_state=1)

scores = []

for train_idx, val_idx in kfold.split(df_full_train):
    df_train = df_full_train.iloc[train_idx]
    df_val = df_full_train.iloc[val_idx]

    y_train = df_train.churn.values
    y_val = df_val.churn.values

    dv, model = train(df_train, y_train, C=C)
    y_pred = predict(df_val, dv, model)

    auc = roc_auc_score(y_val, y_pred)
    scores.append(auc)

print('C=%s %.3f +- %.3f' % (C, np.mean(scores), np.std(scores)))

C=1.0 0.842 +- 0.007


In [29]:
scores

[0.8443375120691413,
 0.8449977651966627,
 0.8333012241621514,
 0.8347768913474451,
 0.8518166985660947]

In [30]:
dv, model = train(df_full_train, df_full_train.churn.values, C=1.0)
y_pred = predict(df_test, dv, model)

y_test = df_test.churn.values
auc = roc_auc_score(y_test, y_pred)
auc

0.8584032088573997

### Saving with pickle

Pickle is a standard Python library for serializing objects: it writes a Python object into a binary file, and can read it back later. **We save both the vectorizer and the model together** as a tuple:

In [31]:
output_file = f'model_C={C}.bin'

with open(output_file, 'wb') as f_out:
    pickle.dump((dv, model), f_out)

The `'wb'` mode means write-binary: pickle produces binary data, so we must not open the file in text mode.  
We name the file with the value of `C` we used, because after tuning in the previous module we know this parameter, and it is useful to see it in the filename: `model_C=1.0.bin`.

**The vectorizer has to travel with the model. It was fitted on the training data and it "remembers" which categorical values map to which feature columns. Without it, we cannot prepare a new customer record the same way.**

### Loading the model

To use the model (for example, from a web service) we load the file back:

In [32]:
with open('model_C=1.0.bin', 'rb') as f_in:
    dv, model = pickle.load(f_in)

Here `'rb'` is read-binary. `pickle.load` returns the tuple we saved, and we unpack it into dv and model. From this point we can score customers without running any training code.

One warning: never unpickle a file from a source you do not trust. Pickle files can contain code, and loading one executes it. Only load files you created yourself or that come from someone you trust.

### Scikit-Learn Pipelines

**It's not convenient to deal with two objects: `dv` and `model`. Let's combine them into one using scikit-learn's pipelines which are more modern.**

In [40]:
pipeline = make_pipeline(
    DictVectorizer(sparse=False),
    LogisticRegression(
        solver="liblinear",
        C=1.0,
        max_iter=1000,
        random_state=42
    )
)

X_train = df_full_train[categorical + numerical].to_dict(orient="records")
y_train = df_full_train["churn"]

pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('dictvectorizer', ...), ('logisticregression', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
,"sparse sparse: bool, default=TrueWhether transform should produce scipy.sparse matrices.",False
,"dtype dtype: dtype, default=np.float64The type of feature values. Passed to Numpy array/scipy.sparse matrixconstructors as the dtype argument.",<class 'numpy.float64'>
,"separator separator: str, default=""=""Separator string used when constructing new features for one-hotcoding.",'='
,"sort sort: bool, default=TrueWhether ``feature_names_`` and ``vocabulary_`` should besorted when fitting.",True
Name,Type,Value


In [44]:
datapoint = {
    'gender': 'female',
    'seniorcitizen': 0,
    'partner': 'yes',
    'dependents': 'no',
    'phoneservice': 'no',
    'multiplelines': 'no_phone_service',
    'internetservice': 'dsl',
    'onlinesecurity': 'no',
    'onlinebackup': 'yes',
    'deviceprotection': 'no',
    'techsupport': 'no',
    'streamingtv': 'no',
    'streamingmovies': 'no',
    'contract': 'month-to-month',
    'paperlessbilling': 'yes',
    'paymentmethod': 'electronic_check',
    'tenure': 1,
    'monthlycharges': 29.85,
    'totalcharges': 29.85
}

pipeline.predict_proba(datapoint)[0, 1]

np.float64(0.6433010959417496)

In [ ]:
# Let's compare the prediction of the model trained using the pipeline, against the model trained the old way
X = dv.transform(datapoint)
model.predict_proba(X)[0, 1]

np.float64(0.6275953527536646)

Why are the results different between the models? Because they were trained with different `LogisticRegression` settings.  
The earlier model uses the default `lbfgs` solver with `max_iter=3000`.  
The pipeline uses `liblinear` with `max_iter=1000` and `random_state=42`.  
Different solvers can learn slightly different coefficients, so their predicted probabilities need not match.  
But training and optimizing predictions is not the objective of this chapter. We want to learn how to deploy models.

### Turning the notebook into a script

The last step is moving the training code out of the notebook into a plain Python script.  
It contains the same steps we developed in the notebook.

A script is what we will actually run on a server: it is reproducible, it can be executed by a scheduler or a CI job, and it does not require a running Jupyter instance.

To turn a notebook into a script we can use `jupyter nbconvert --to script <path_to_ipynb> --output=<name>`:

In [51]:
!jupyter nbconvert --to script 05-notes.ipynb --output train

[NbConvertApp] Converting notebook 05-notes.ipynb to script
[NbConvertApp] Writing 10015 bytes to train.py


Now we move to [train.py](https://github.com/k-candidate/Machine-Learning-Zoomcamp-2026/blob/main/05_-_Deployment/train.py) and edit it there. Read that script to be able to follow.

## 3. Web services: introduction

In this section we learn what a web service is and write our first one: a tiny Flask app that answers a ping request with "PONG".

### What is a web service?

A web service is a way for applications to talk to each other over HTTP (the same protocol your browser uses to load websites).  
One application (a client) sends a request to an address (a URL), and the other application (the server) sends back a response.

The client says not only where to send the request, but also what it wants to do: this is the HTTP method.  
The most common ones:
- GET: retrieve something. Opening a page in the browser sends a GET request; searching for a cat picture on Google is a GET request too.
- POST: send data to the server, for example the signup form with your name and email, or a customer record we want to score.
- PUT: similar to POST, but the request says where the data should go.
- DELETE: remove something from the server.

For our model deployment we care about two of them: GET for a health-check endpoint, and POST for sending customer data to the model.

### A ping/pong service in Flask

Flask is a Python framework for building web services.

Read [ping-flask.py](https://github.com/k-candidate/Machine-Learning-Zoomcamp-2026/blob/main/05_-_Deployment/ping-flask.py).

### Testing the Flask ping/pong service

Run the service:
```bash
python ping.py
```

Then from another terminal, query with curl:
```bash
curl http://localhost:9696/ping
```

The response is `PONG`

Opening `http://localhost:9696/ping` in a browser works as well because the browser also sends a GET request.

### A ping/pong service in FastAPI

FastAPI is a more modern Python framework for building web services.

Read [ping-fastapi.py](https://github.com/k-candidate/Machine-Learning-Zoomcamp-2026/blob/main/05_-_Deployment/ping-fastapi.py).

It's pretty much similar to Flask with the added benefir that we can go to http://localhost:9696/docs in our browser to test the endpoint in there.

## 4. Serving the churn model with FastAPI

In this section we wrap the churn model into a FastAPI web service: it loads the pickled model, listens for POST requests with customer data, and replies with the churn probability.

### The web service

We take the ping app from the previous section and extend it. The full service is in [predict.py](https://github.com/k-candidate/Machine-Learning-Zoomcamp-2026/blob/main/05_-_Deployment/predict.py). Go and read it.

While developing it we can use:
```bash
uvicorn predict:app --host 0.0.0.0 --port 9696 --reload
```

### Testing the service

Run the service with `python predict.py`.

We can test it via the browser in https://localhost:9696/docs because it allows us to use POST requests.

We can also use a small client script. Read [test.py](https://github.com/k-candidate/Machine-Learning-Zoomcamp-2026/blob/main/05_-_Deployment/test.py) which sends a request with the `requests` library.

## 5. Environment Management

For env mgmt, you can use whatever you want. I use `uv` as you already know if you are following along this series of lessons.

If you want to learn more about `uv`, you can go to https://k-candidate.github.io/2025/09/01/rustification-of-python-uv-ruff-ty.html.

## 6. Docker

Read [Dockerfile](https://github.com/k-candidate/Machine-Learning-Zoomcamp-2026/blob/main/05_-_Deployment/Dockerfile) and [bake-file.hcl](https://github.com/k-candidate/Machine-Learning-Zoomcamp-2026/blob/main/05_-_Deployment/bake-file.hcl).

Build and push the image by running:
```bash
docker buildx bake
```

Reclaim the cache space with `docker buildx prune`. You can see what you have in there with `docker buildx du`.

Run the image:
```bash
docker run -it --rm -p 9696:9696 docker.io/kcandidate/predict-churn:v1
```

## 07. Deployment

Once the application is dockerized, you can deploy it in any system that supports containers: a bare-metal server, K8s, AWS EKS, AWS ECS, Azure AKS, Azure Container Apps, GCP GKS, GCP Cloud Run ...